# Capítulo 21 - MLOps para LLMs On-Premise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap21_mlops_llm_on_premise.ipynb)

Notebook **prático e autoral**: práticas de MLOps do **AI-Orchestrator** — configuração como código (12-Factor), gates de avaliação, observabilidade com *graceful degradation* e seleção de modelo por ambiente. Executável, sem dependências externas.

---

## Atribuição

Material **autoral** do AI-Orchestrator, por Allan Eric Jepsen. Espelha: `gateway/config.py`, `evals/`, `gateway/observability` e `docker-compose.yml`. Texto: `livro/cap21_mlops_llm_on_premise.md`.

---

## Parte 1 — Configuração como código (12-Factor)

Tudo é sobrescrevível por **variável de ambiente**, carregado uma vez no boot num `dataclass` **imutável** (`frozen=True`). Espelha `gateway/config.py`. Imutabilidade evita *config drift* em runtime.

In [1]:
import os
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Settings:
    model: str = field(default_factory=lambda: os.environ.get("MODEL", "qwen2.5:7b-instruct-q4_K_M"))
    semantic_threshold: float = field(default_factory=lambda: float(os.environ.get("SEMANTIC_THRESHOLD", "0.92")))
    rate_limit_per_hour: int = field(default_factory=lambda: int(os.environ.get("RATE_LIMIT_PER_HOUR", "10")))

s = Settings()
print("defaults:", s.model, s.semantic_threshold, s.rate_limit_per_hour)
os.environ["SEMANTIC_THRESHOLD"] = "0.85"
print("override por env:", Settings().semantic_threshold)
try:
    s.model = "outro"          # frozen => erro
except Exception as e:
    print("imutavel:", type(e).__name__)

defaults: qwen2.5:7b-instruct-q4_K_M 0.92 10
override por env: 0.85
imutavel: FrozenInstanceError


## Parte 2 — Gates de avaliação (o que bloqueia deploy)

O projeto tem 3 famílias de eval com **gates**: domínios (≥80% acerto), roteamento (≥90%) e injection (0 vazamentos). Qualquer falha **bloqueia o deploy**. Veja `evals/eval_domains.py`, `eval_routing.py`, `eval_injection.py`.

In [2]:
def gate(name, value, threshold, mode=">="):
    ok = value >= threshold if mode == ">=" else value <= threshold
    print(f"[{'PASS' if ok else 'FAIL'}] {name:18} {value:.2f} {mode} {threshold}")
    return ok

resultados = {
    "domain_accuracy": (0.86, 0.80, ">="),
    "routing_accuracy": (0.92, 0.90, ">="),
    "injection_leaks": (0.0, 0.0, "<="),
}
todos_ok = all(gate(n, v, t, m) for n, (v, t, m) in resultados.items())
print("\nDEPLOY", "LIBERADO" if todos_ok else "BLOQUEADO")

[PASS] domain_accuracy    0.86 >= 0.8
[PASS] routing_accuracy   0.92 >= 0.9
[PASS] injection_leaks    0.00 <= 0.0

DEPLOY LIBERADO


## Parte 3 — Observabilidade com *graceful degradation*

O projeto usa **Langfuse** (dual-mode: Cloud ou self-hosted). Regra de ouro: se a observabilidade falhar, **nunca** derrubar a request — degrada e segue. Espelha o padrão de wrapper do gateway.

In [3]:
import logging
logging.basicConfig(level=logging.WARNING)
log = logging.getLogger("obs")

class Telemetry:
    def __init__(self, sink):
        self._sink = sink
    def trace(self, event, **kw):
        try:
            self._sink(event, kw)          # no projeto: Langfuse client
        except Exception as e:             # graceful: nunca propaga
            log.warning("telemetria falhou (%s) - seguindo sem trace", e)

def sink_quebrado(event, kw):
    raise RuntimeError("Langfuse indisponivel")

t = Telemetry(sink_quebrado)
t.trace("chat_request", domain="vendas", latency_ms=120)
print("request seguiu normalmente mesmo com a telemetria fora.")

request seguiu normalmente mesmo com a telemetria fora.


## Parte 4 — Model serving: seleção por ambiente

O modelo é escolhido por variável de ambiente (`MODEL`), servido pelo **Ollama**. Trade-off VRAM × capacidade na RTX 3060.

In [4]:
CATALOGO = {
    "qwen2.5:3b-instruct-q4_K_M": {"vram_gb": 3, "nota": "rapido"},
    "qwen2.5:7b-instruct-q4_K_M": {"vram_gb": 6, "nota": "equilibrado (default)"},
    "qwen2.5:14b-instruct-q4_K_M": {"vram_gb": 10, "nota": "capaz, exige mais VRAM"},
}
VRAM_DISPONIVEL = 8

def escolher(model_env):
    spec = CATALOGO.get(model_env)
    if spec is None:
        return "modelo desconhecido"
    cabe = spec["vram_gb"] <= VRAM_DISPONIVEL
    return f"{model_env} -> {spec['nota']} | cabe em {VRAM_DISPONIVEL}GB? {cabe}"

for m in CATALOGO:
    print(escolher(m))

qwen2.5:3b-instruct-q4_K_M -> rapido | cabe em 8GB? True
qwen2.5:7b-instruct-q4_K_M -> equilibrado (default) | cabe em 8GB? True
qwen2.5:14b-instruct-q4_K_M -> capaz, exige mais VRAM | cabe em 8GB? False


## Parte 5 — Maturidade MLOps

| Nível | Característica | AI-Orchestrator |
|---|---|---|
| 0 | Manual, notebooks soltos | — |
| 1 | Containerizado + testes + evals + config-as-code | **aqui** (c/ elementos do N2) |
| 2 | CI/CD automatizado, retraining | parcial (pipeline ideal documentado) |
| 3 | Full automation, monitoring contínuo | roadmap |

O projeto está no **Nível 1 com elementos de Nível 2** (gates de eval definidos; CI/CD documentado mas ainda manual).

## Conclusão e mapa para o código

| Etapa | Arquivo real |
|---|---|
| Config 12-Factor (frozen) | `gateway/config.py` |
| Gates de avaliação | `evals/eval_domains.py`, `eval_routing.py`, `eval_injection.py`, `eval_semiose.py` |
| Observabilidade dual-mode | integração Langfuse |
| Serving por env | `docker-compose.yml`, Ollama |

**Princípio:** reprodutibilidade + gates que bloqueiam deploy + degradação graceful. Texto: `livro/cap21_mlops_llm_on_premise.md`.